<a href="https://colab.research.google.com/github/emily-wilson/cs229-final-proj/blob/main/cs229_prediction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
!pip install librosa
!pip install -U -q transformers
!pip install torch
!pip install evaluate
!pip install jiwer ## evaluate dependency

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.7 MB/s eta 0:00:00


In [ ]:
#@title Imports
import librosa
import numpy as np
import json
from tqdm.notebook import trange
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, WhisperTokenizer, WhisperFeatureExtractor
from datasets import load_dataset
import torch
import math

In [ ]:
## Set up HuggingFace to cache to GoogleDrive (from my Spr2025 CS224S HW4)
# You need to run this code for every colab runtime
from google.colab import drive
import os

drive.mount('/content/drive')

# creating a folder for saving the cache
drive_cache_dir = "/content/drive/MyDrive/hf_datasets_cache"
os.makedirs(drive_cache_dir, exist_ok=True)

# changing hf cache directory
os.environ["HF_DATASETS_CACHE"] = drive_cache_dir
os.environ["HF_HOME"] = drive_cache_dir

Mounted at /content/drive


In [ ]:
MODEL_ID = "openai/whisper-large-v3"

## Set up Google Drive dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
APROCSA_DATASET_LOC = "cs229_final_proj/aprocsa_dataset"
dataset_filename = f'drive/MyDrive/{APROCSA_DATASET_LOC}/dataset.json'

# Using Whisper model for prediction and pretraining

In [ ]:
#@title Load dataset
dataset = load_dataset("json", data_files=dataset_filename, field="dataset", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
MAX_LEN = 15 * 16000
def filter_dataset(audio):
  return len(audio["array"]) <= MAX_LEN

dataset = dataset.filter(filter_dataset, include_columns=["audio"])

In [ ]:
#@title Baseline prediction with Whispr model
## Code is based on model card for Whispr on HuggingFace (https://huggingface.co/openai/whisper-large-v3)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(MODEL_ID)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

results = []
for result in pipe(dataset, batch_size=2, generate_kwargs={"language": "english"}):
  results.append(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


TypeError: We expect a numpy ndarray or torch tensor as input, got `<class 'datasets.arrow_dataset.Dataset'>`

In [ ]:
from jiwer import wer, cer
wers = []
cers = []
for i in range(len(results)):
  reference = test_data[i]["transcript"]["transcript"].lower()
  hypothesis = clean_output(results[i]["text"])

  wers.append(wer(reference, hypothesis))
  cers.append(cer(reference, hypothesis))

wers = np.array(wers)
cers = np.array(cers)

print("Average WER: ", np.mean(wers))
print("Average CER: ", np.mean(cers))

In [ ]:
import pandas as pd
output_csv_path = "drive/MyDrive/cs229_final_proj/baseline_whisper_preds.csv"

In [ ]:
#@title Write results to CSV
data = {
    "hypothesis": [result["text"] for result in results],
    "reference": [d["transcript"]["transcript"] for d in test_data],
    "wer": wers,
    "cer": cers
}

df = pd.DataFrame(data)
df.to_csv(output_csv_path, index=False)

In [ ]:
#@title Loading data from csv
df = pd.read_csv(output_csv_path)

In [ ]:
#@title Reformat csv
from jiwer import wer, cer, mer

hypotheses = df["hypothesis"]
references = df["reference"]

hypotheses = [h.lower() for h in hypotheses]
references = [r.lower() for r in references]

wers = [wer(references[i], hypotheses[i]) for i in range(len(hypotheses))]
cers = [cer(references[i], hypotheses[i]) for i in range(len(hypotheses))]
mers = [mer(references[i], hypotheses[i]) for i in range(len(hypotheses))]

print("Average WER: ", np.mean(np.array(wers)))
print("Average CER: ", np.mean(np.array(cers)))
print("Average MER: ", np.mean(np.array(mers)))

data = {
    "hypothesis": hypotheses,
    "reference": references,
    "wer": wers,
    "cer": cers,
    "mer": mers,
}
df = pd.DataFrame(data)

In [ ]:
df.to_csv(output_csv_path, index=False)

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()
df

In [ ]:
wer_cer_diff = [abs(wers[i] - cers[i]) for i in range(len(wers))]
diff_data = {
    "hypothesis": [result["text"].strip() for result in results],
    "reference": [d["transcript"]["transcript"] for d in test_data],
    "wer": wers,
    "cer": cers,
    "diff": wer_cer_diff
}

df2 = pd.DataFrame(diff_data)
df2

# Model improvements for Whisper ASR

The code here is based on the HuggingFace blog post for how to finetune the Whisper model

In [ ]:
#@title Set up Weights and Biases for finetuning logging
# (from Weights and Biases blog post https://wandb.ai/ayush-thakur/huggingface/reports/How-To-Fine-Tune-Hugging-Face-Transformers-on-a-Custom-Dataset--Vmlldzo0MzQ2MDc)
# Install Weights and Biases
!pip install wandb -q

# Import wandb
import wandb

# Login with your authentication key
wandb.login()

# setup wandb environment variables
%env WANDB_ENTITY=emilywilson6263/emilywilson6263-stanford-university
%env WANDB_PROJECT=cs-229-final-project


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:



KeyboardInterrupt: 

In [ ]:
#@title Prepare feature extractor
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)

In [ ]:
#@title Load dataset

dataset = load_dataset("json", data_files=dataset_filename, field="dataset", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
#@title Remove special characters that will be added back in post-processing
import re

encodings_to_remove = [r"\[\+ gram\]", "\[/\]", "&=shrugs", "\[//\]", "&\-", "&\+", "\[\+ exc\]", "\+<", "\[\+ jar\]"]
def remove_some_special_characters(batch):
  transcript = re.sub(r"[<>]", "", batch["transcript"]["transcript"])
  for encoding in encodings_to_remove:
    transcript = re.sub(encoding, "", transcript)
  batch["transcript"] = transcript
  return batch

dataset = dataset.map(remove_some_special_characters)

<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\-'
<>:4: SyntaxWarning: invalid escape sequence '\+'
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\+'
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\-'
<>:4: SyntaxWarning: invalid escape sequence '\+'
<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\+'
<>:4: SyntaxWarning: invalid escape sequence '\['
/tmp/ipython-input-3113689591.py:4: SyntaxWarning: invalid escape sequence '\['
  encodings_to_remove = [r"\[\+ gram\]", "\[/\]", "&=shrugs", "\[//\]", "&\-", "&\+", "\[\+ exc\]", "\+<", "\[\+ jar\]"]
/tmp/ipython-input-3113689591.py:4: SyntaxWarning: invalid escape sequence '\['
  encodings_to_remo

Map:   0%|          | 0/1645 [00:00<?, ? examples/s]

In [ ]:
#@title Build string replacement map
LONG_PAUSE_STRING = r"\(...\)"
LONG_PAUSE_TOKEN = "<long_pause>"
SHORT_PAUSE_STRING = r"\(.\)"
SHORT_PAUSE_TOKEN = "<short_pause>"
MEDIUM_PAUSE_STRING = r"\(..\)"
MEDIUM_PAUSE_TOKEN = "<medium_pause>"
GES_STRING = r"&ges=\w+\s"
GES_TOKEN = "<gesture>"
UNK_TOKEN = "xxx"
COMMENT_TOKEN = r"\[% \w+\]"
COMMENT_REPLACE = ""
TRAIL_OFF_STRING = r"\+..."
TRAIL_OFF_QUESTION_STRING = r"\+..?"
TRAIL_OFF_TOKEN = "<trail_off>"
LAUGHTER_STRING = "&=laughs"
LAUGHTER_TOKEN = "<laughter>"

STRING_REPLACEMENT_MAP = {
    LONG_PAUSE_STRING: LONG_PAUSE_TOKEN,
    MEDIUM_PAUSE_STRING: MEDIUM_PAUSE_TOKEN,
    SHORT_PAUSE_STRING: SHORT_PAUSE_TOKEN,
    GES_STRING: GES_TOKEN,
    COMMENT_TOKEN: COMMENT_REPLACE,
    TRAIL_OFF_STRING: TRAIL_OFF_TOKEN,
    LAUGHTER_STRING: LAUGHTER_TOKEN,
    TRAIL_OFF_QUESTION_STRING: TRAIL_OFF_TOKEN,
}

In [ ]:
#@title Reformat special characters
def reformat_chat_transcription_annotations(batch):
  for string, replace in STRING_REPLACEMENT_MAP.items():
    batch["transcript"] = re.sub(string, replace, batch["transcript"])

dataset = dataset.map(reformat_chat_transcription_annotations)


Map:   0%|          | 0/1622 [00:00<?, ? examples/s]

NameError: name 're' is not defined

In [ ]:
#@title Prepare the tokenizer for the special characters in the CHAT file format
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID)

special_tokens_dict = {}
special_tokens_list = list(STRING_REPLACEMENT_MAP.values())
for i in range(len(special_tokens_list)):
  special_tokens_dict[len(tokenizer) + i] = special_tokens_list[i]

tokenizer.add_tokens(list(special_tokens_dict.values()))

In [ ]:
print(tokenizer.vocab_size)

In [ ]:
## Only need to run this once
!curl -O f "drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/aprocsa_dataset/merges.txt" https://huggingface.co/openai/whisper-large/resolve/main/merges.txt

In [ ]:
#@title Set up processor
from transformers import WhisperProcessor

processor = WhisperProcessor(feature_extractor, tokenizer)

## Prepare dataset

In [ ]:
MAX_LEN = 15*16000
def filter_dataset(audio):
  return len(audio["array"]) <= MAX_LEN

def prepare_dataset(batch):
    audio = batch["audio"]["array"]
    ## Audio was resampled in dataset creation so it is sampled at 16kHz
    batch["audio"] = audio

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(batch["audio"], sampling_rate=16000).input_features[0]

    # encode target text to label ids
    # with processor.as_target_processor():
    batch["labels"] = tokenizer(batch["transcript"]).input_ids
    return batch

In [ ]:
dataset = dataset.filter(filter_dataset, input_columns=["audio"])
dataset = dataset.map(prepare_dataset, num_proc=1)

splits = dataset.train_test_split(test_size=0.1)
train_data = splits["train"]
test_data = splits["test"]

splits = train_data.train_test_split(test_size=0.2)
train_data = splits["train"]
valid_data = splits["test"]

Filter:   0%|          | 0/1622 [00:00<?, ? examples/s]

Map:   0%|          | 0/1622 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [ ]:
#@title Set up model checkpoint using whisper-large-v3
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

In [ ]:
model.resize_token_embeddings(len(tokenizer))

In [ ]:
# model.config.vocab_size = len(all_tokens)
model.config.forced_decoder_ids = None

In [ ]:
#@title Set up data collator
## Note this comes directly from the HuggingFace blog post of how to fine-tune Whisper
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollator:
  processor: Any
  decoder_start_token_id: int

  def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
    # split inputs and labels since they have to be of different lengths and need different padding methods
    # first treat the audio inputs by simply returning torch tensors
    input_features = [{"input_features": feature["input_features"]} for feature in features]
    batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

    # get the tokenized label sequences
    label_features = [{"input_ids": feature["labels"]} for feature in features]
    # pad the labels to max length
    labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

    # replace padding with -100 to ignore loss correctly
    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

    # if bos token is appended in previous tokenization step,
    # cut bos token here as it's append later anyways
    if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
        labels = labels[:, 1:]

    batch["labels"] = labels

    return batch

In [ ]:
data_collator = DataCollator(
  processor=processor,
  decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
#@title Define metrics
## We are using CER (character error rate) here since we are fine-tuning the model
## to predict the special characters of the CHAT transcript
import evaluate

metric = evaluate.load("cer")

## Also copied from HuggingFace blog post
def compute_metrics(pred):
  pred_ids = pred.predictions
  label_ids = pred.label_ids

  ## TODO: should we do this? for us sometimes gaps may have semantic meaning
  # replace -100 with the pad_token_id
  label_ids[label_ids == -100] = tokenizer.pad_token_id

  # we do not want to group tokens when computing the metrics
  pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
  label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

  cer = 100 * metric.compute(predictions=pred_str, references=label_str)

  return {"cer": cer}

In [ ]:
#@title Set up training args and trainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, logging

output_dir = f'drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/finetuning/'

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,  # change to a repo name of your choice
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=1600,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=280,
    save_steps=400,
    eval_steps=400,
    logging_steps=10,
    report_to=None,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    push_to_hub=False,
    disable_tqdm=False,
    torch_empty_cache_steps=5
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=valid_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
)

logging.enable_progress_bar()
logging.set_verbosity_info()

In [ ]:
!rm ~/.netrc
!wandb login

In [ ]:
torch.cuda.empty_cache()

In [ ]:
#@title Start training!
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

torch.cuda.empty_cache()

trainer.train()

## Evaluate model

### Load and prepare dataset

In [ ]:
dataset = load_dataset("json", data_files=dataset_filename, field="dataset", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
MAX_LEN = 15*16000
def filter_dataset(audio):
  return len(audio["array"]) <= MAX_LEN

def process_dataset_for_evaluation(batch):
  batch["audio"] = batch["audio"]["array"]
  return batch

In [ ]:
dataset = dataset.filter(filter_dataset, input_columns=["audio"])
dataset = dataset.map(process_dataset_for_evaluation)

splits = dataset.train_test_split(test_size=0.1)
train_data = splits["train"]
test_data = splits["test"]

splits = train_data.train_test_split(test_size=0.2)
train_data = splits["train"]
valid_data = splits["test"]

Filter:   0%|          | 0/1645 [00:00<?, ? examples/s]

Map:   0%|          | 0/1622 [00:00<?, ? examples/s]

In [ ]:
model_checkpoint = "drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/finetuning/checkpoint-1200"

## Predict

In [ ]:
## Code is based on model card for Whispr on HuggingFace (https://huggingface.co/openai/whisper-large-v3)
## TODO: run this on the test dataset first to get a baseline
from transformers import WhisperForConditionalGeneration

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = WhisperForConditionalGeneration.from_pretrained(model_checkpoint)
model.to(device)

tokenizer = WhisperTokenizer.from_pretrained(model_checkpoint)
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)

results = []
references = []
for batch in valid_data.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.extend(predictions)
  references.extend(batch["transcript"])


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Batching examples:   0%|          | 0/292 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:655: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 

In [ ]:
#@title Replace training tokens with CHAT format characters
import re

for i in range(len(results)):
  for key, value in STRING_REPLACEMENT_MAP.items():
    if value != "" and re.match(value, results[i]):
      results[i] = re.sub(value, key, results[i])
print(results)

[' mhm .', ' alright .', ' டே hello .', ' mhm .', ' Yep .', " I don't have a  uh uh um maid to run this house .", ' So ‡ okay .', " look at everything's that's happening .", " I don't know why .", ' close . ', ' . ', ' Beraia.', ' The universe knows sometimes .', " I don't know .", ' yeah ‡ drunk . ', ' take a little bit of time to look at those pictures .', ' I samo kao da ste ube, počekajte učištajne uči koje se može, sato really wanna hear your language .', ' Og så lever hun op i atticen .', ' okay .', ' they  they tried to uh get  stick it in their feet to the slipper .', ' But a lot.)ha.) s changed .', ' and that was it .', " That's uh very current and uh uh very important at the moment .", " I don't know why .", ' Ok.', " I'm just  I can't imagine having to face a bill like that and having the uncertainty of how much would be covered .", " uh I  I don't know .", ' So you do those ?', ' hm mhm .', ' Mhm .', ' okay .', " but it's good &-ges .", ' and Cinderella came .', ' The um yo

In [ ]:
import csv

with open('drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/transcription_results_aprocsa.csv', 'w', newline='\n') as csvfile:
  writer = csv.DictWriter(csvfile, fieldnames=["hypothesis", "reference"])

  writer.writeheader()
  for i in range(len(results)):
    writer.writerow({"hypothesis": results[i], "reference": references[i]["transcript"]})


## Post-processing predicted text

Use OpenAI to take predicted transcripts and format them to reintroduce CHAT transcription format tokens like repetitions, fragments, retracing, and filler words.

In [ ]:
!pip install openai

In [ ]:
FILLER_WORD_PREFIX = "&-"
FRAGMENT_PREFIX = "&+"
JARGON_POSTFIX = "[+ jar]"
REPETITION_PREFIX = "[/]"
RETRACING_PREFIX = "[//]"

In [ ]:
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('229-open-ai-api-key')

client = OpenAI()

FILLER_WORDS_INSTRUCTIONS = f'Reformat the given string to add the prefix {FILLER_WORD_PREFIX} to all common filler words (e.g. "um", "er"). Return ONLY the reformatted string and change nothing else besides adding the prefix'

for i in range(len(results)):
  response = client.responses.create(
      model="gpt-5.1",
      instructions=FILLER_WORDS_INSTRUCTIONS,
      input=results[i]
  )

  print(response.output_text)


REPETITION_WORDS_INSTRUCTIONS = f'Reformat the given string to add the token {REPETITION_PREFIX} before any repeated words or sounds. For example the string "b b ball" should be formatted to "[/] b [/] b [/] ball'
for i in range(len(results)):
  response = client.responses.create(
      model="gpt-5.1",
      instructions=REPETITION_WORDS_INSTRUCTIONS,
      input=results[i]
  )

  print(response.output_text)

SecretNotFoundError: Secret 229-open-ai-api-key does not exist.

## Compute metrics

In [ ]:
import csv

reference = []
hypothesis = []
with open('drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/transcription_results_aprocsa.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        reference.append(row["reference"])
        hypothesis.append(row["hypothesis"])

In [ ]:
from jiwer import wer, cer
wers = []
cers = []
for i in range(len(hypothesis)):
  wers.append(wer(reference[i], hypothesis[i]))
  cers.append(cer(reference[i], hypothesis[i]))

wers = np.array(wers)
cers = np.array(cers)

print("Average WER: ", np.mean(wers))
print("Average CER: ", np.mean(cers))

Average WER:  0.38583618995741886
Average CER:  0.27915500552364264


In [ ]:
import pandas as pd
output_csv_path = "drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/finetuned_whisper_metrics.csv"

In [ ]:
#@title Write results to CSV
data = {
    "hypothesis": hypothesis,
    "reference": reference,
    "wer": wers,
    "cer": cers
}

df = pd.DataFrame(data)
df.to_csv(output_csv_path, index=False)

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()
df

,hypothesis,reference,wer,cer
0,mhm .,mhm .,0.000000,0.000000
1,alright .,alright .,0.000000,0.000000
2,டே hello .,"+"" hey ‡ hello .",0.600000,0.500000
3,mhm .,mhm . [+ exc],0.500000,0.615385
4,Yep .,yep .,0.500000,0.200000
...,...,...,...,...
287,and the the the uh lamp f fell over and stu...,&=laughs and the [/] the [/] the &-uh lamp &+f...,0.333333,0.279412
288,What can you do ?,and &-um (..) &-um (.) what can you do ?,0.600000,0.600000
289,I don't know .,I don't know .,0.000000,0.000000
290,mhm .,+< mhm .,0.333333,0.375000


# Baseline Aphasia Classifiers from CLAN Features of Manual Transcription


In [ ]:
#@title Baseline Logistic Regression
import pandas as pd

csv_path = f'drive/MyDrive/clan_features_with_labels.csv'
df = pd.read_csv(csv_path)
df.head()

,File,Language,Corpus,Code,Age,Sex,Group,Role,Duration_(sec),Total_Utts,...,Participant,Word comprehension,Sentence comprehension,Word finding,Grammatical construction,Speech motor programming,Repetition,Reading,Overall,Overall communication impairment
0,aprocsa1554a__G=Cat.cha,eng,APROCSA,PAR,46;00.,female,aphasia,Participant,0,6,...,1554,10.0,9.58,8.0,5.13,7.5,7.08,8.75,7.96,2.0
1,aprocsa1554a__G=Cinderella.cha,eng,APROCSA,PAR,46;00.,female,aphasia,Participant,0,15,...,1554,10.0,9.58,8.0,5.13,7.5,7.08,8.75,7.96,2.0
2,aprocsa1554a__G=Cinderella_Intro.cha,eng,APROCSA,PAR,46;00.,female,aphasia,Participant,0,3,...,1554,10.0,9.58,8.0,5.13,7.5,7.08,8.75,7.96,2.0
3,aprocsa1554a__G=Important_Event.cha,eng,APROCSA,PAR,46;00.,female,aphasia,Participant,0,130,...,1554,10.0,9.58,8.0,5.13,7.5,7.08,8.75,7.96,2.0
4,aprocsa1554a__G=Sandwich.cha,eng,APROCSA,PAR,46;00.,female,aphasia,Participant,0,3,...,1554,10.0,9.58,8.0,5.13,7.5,7.08,8.75,7.96,2.0


In [ ]:
from pathlib import Path

text_dir = Path("drive/MyDrive/par_text")  # change this if needed

transcripts = []
for i, row in df.iterrows():
    fname = row["File"].replace(".cha", ".txt")
    text_path = text_dir / fname

    if text_path.exists():
        transcripts.append(text_path.read_text())
    else:
        transcripts.append("")

df["transcript_text"] = transcripts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]
clan_X = df[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
text_X = vectorizer.fit_transform(df["transcript_text"])

scaler = StandardScaler()
clan_X_scaled = scaler.fit_transform(clan_X)

X = hstack([clan_X_scaled, text_X])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = df["Overall communication impairment"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"  # important for small dataset
)

logreg.fit(X_train, y_train)

preds = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.9090909090909091
              precision    recall  f1-score   support

           1       1.00      0.50      0.67         2
           2       0.88      1.00      0.93         7
           3       1.00      1.00      1.00         2

    accuracy                           0.91        11
   macro avg       0.96      0.83      0.87        11
weighted avg       0.92      0.91      0.90        11



In [ ]:
#@title Baseline XG Boost

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load data
df = pd.read_csv('drive/MyDrive/clan_features_with_labels.csv')

# Load transcripts
text_dir = Path("drive/MyDrive/par_text")
transcripts = []
for i, row in df.iterrows():
    fname = row["File"].replace(".cha", ".txt")
    text_path = text_dir / fname
    if text_path.exists():
        transcripts.append(text_path.read_text())
    else:
        transcripts.append("")

df["transcript_text"] = transcripts

In [ ]:
# Prepare labels (convert to 0-indexed)
y = df["Overall communication impairment"].astype(int) - 1

# Train/test split
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

df_train = df.loc[train_idx]
df_test = df.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# Extract features
exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]

# CLAN features
clan_X_train = df_train[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_test = df_test[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

# Fit scaler on train only
scaler = StandardScaler()
clan_X_train_scaled = scaler.fit_transform(clan_X_train)
clan_X_test_scaled = scaler.transform(clan_X_test)

# Fit vectorizer on train only
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
text_X_train = vectorizer.fit_transform(df_train["transcript_text"])
text_X_test = vectorizer.transform(df_test["transcript_text"])

# Combine features
X_train = hstack([clan_X_train_scaled, text_X_train])
X_test = hstack([clan_X_test_scaled, text_X_test])

print(f"Dataset size: {len(df)} total, {len(train_idx)} train, {len(test_idx)} test")
print(f"Class distribution in test: {y_test.value_counts().sort_index().to_dict()}")

Dataset size: 54 total, 43 train, 11 test
Class distribution in test: {0: 2, 1: 7, 2: 2}


In [ ]:
# XGBoost model with heavy regularization
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    max_depth=2,
    n_estimators=30,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42
)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, preds_xgb)
print(f"Test Accuracy: {acc_xgb:.3f}")
print(classification_report(y_test, preds_xgb))

Test Accuracy: 0.636
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.64      1.00      0.78         7
           2       0.00      0.00      0.00         2

    accuracy                           0.64        11
   macro avg       0.21      0.33      0.26        11
weighted avg       0.40      0.64      0.49        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#@title Baseline Naive Bayes

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
import numpy as np

In [ ]:
# Load data
df = pd.read_csv('drive/MyDrive/clan_features_with_labels.csv')

# Load transcripts
text_dir = Path("drive/MyDrive/par_text")
transcripts = []
for i, row in df.iterrows():
    fname = row["File"].replace(".cha", ".txt")
    text_path = text_dir / fname
    if text_path.exists():
        transcripts.append(text_path.read_text())
    else:
        transcripts.append("")

df["transcript_text"] = transcripts

In [ ]:
# Prepare features
exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]
y = df["Overall communication impairment"].astype(int) - 1

print(f"Total dataset size: {len(df)}")
print(f"Class distribution:\n{y.value_counts().sort_index()}\n")

# Split data
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

df_train = df.loc[train_idx]
df_test = df.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

Total dataset size: 54
Class distribution:
Overall communication impairment
0     9
1    36
2     9
Name: count, dtype: int64



In [ ]:
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1,2))
text_X_train = vectorizer.fit_transform(df_train["transcript_text"])
text_X_test = vectorizer.transform(df_test["transcript_text"])

nb_text = MultinomialNB()
nb_text.fit(text_X_train, y_train)
preds_text = nb_text.predict(text_X_test)

print(f"Test Accuracy: {accuracy_score(y_test, preds_text):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_text))

Test Accuracy: 0.636

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.64      1.00      0.78         7
           2       0.00      0.00      0.00         2

    accuracy                           0.64        11
   macro avg       0.21      0.33      0.26        11
weighted avg       0.40      0.64      0.49        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
clan_X_full = df[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_full_scaled = StandardScaler().fit_transform(clan_X_full)

print("\n5-Fold CV Accuracy:")
cv_scores_clan = cross_val_score(
    GaussianNB(),
    clan_X_full_scaled,
    y,
    cv=5,
    scoring='accuracy'
)
print(f"Mean: {cv_scores_clan.mean():.3f} (+/- {cv_scores_clan.std():.3f})")
print(f"Individual folds: {cv_scores_clan}")


5-Fold CV Accuracy:
Mean: 0.982 (+/- 0.036)
Individual folds: [0.90909091 1.         1.         1.         1.        ]


In [ ]:
X_train_combined = hstack([clan_X_train_scaled, text_X_train]).toarray()
X_test_combined = hstack([clan_X_test_scaled, text_X_test]).toarray()

nb_combined = GaussianNB()
nb_combined.fit(X_train_combined, y_train)
preds_combined = nb_combined.predict(X_test_combined)

print(f"Test Accuracy: {accuracy_score(y_test, preds_combined):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_combined))

Test Accuracy: 0.727

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.70      1.00      0.82         7
           2       1.00      0.50      0.67         2

    accuracy                           0.73        11
   macro avg       0.57      0.50      0.50        11
weighted avg       0.63      0.73      0.65        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Baseline Dysarthria Classification with TORGO Datset

In [ ]:
#@title Load Dataset

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier


In [ ]:
data_dir = "drive/MyDrive/torgo_processed_data"
X = np.load(f"{data_dir}/X_mfcc.npy")
y = np.load(f"{data_dir}/Y.npy")

print(f"MFCC features: {X.shape}")
print(f"Labels: {y.shape}")

MFCC features: (8019, 128)
Labels: (8019,)


In [ ]:
print(f"\nTotal: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class distribution: {np.bincount(y)}")
print(f"  Class 0 (No dysarthria): {np.sum(y==0)}")
print(f"  Class 1 (Dysarthria): {np.sum(y==1)}")


Total: 8019 samples, 128 features
Class distribution: [5122 2897]
  Class 0 (No dysarthria): 5122
  Class 1 (Dysarthria): 2897


In [ ]:
#@title Baseline Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain samples: {len(X_train)}, Test samples: {len(X_test)}")


Train samples: 6415, Test samples: 1604


In [ ]:
logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",  # Important for imbalanced data
    random_state=42
)

logreg.fit(X_train_scaled, y_train)

y_pred = logreg.predict(X_test_scaled)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                          target_names=['No Dysarthria', 'Dysarthria'],
                          digits=4))

Test Accuracy: 0.9582

Classification Report:
               precision    recall  f1-score   support

No Dysarthria     0.9669    0.9678    0.9673      1025
   Dysarthria     0.9429    0.9413    0.9421       579

     accuracy                         0.9582      1604
    macro avg     0.9549    0.9545    0.9547      1604
 weighted avg     0.9582    0.9582    0.9582      1604



In [ ]:
#@title Baseline Naive Bayes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train: 6415, Test: 1604



In [ ]:
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
preds_nb = nb.predict(X_test_scaled)

print(f"Test Accuracy: {accuracy_score(y_test, preds_nb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds_nb,
                          target_names=['No Dysarthria', 'Dysarthria']))

Test Accuracy: 0.7868

Classification Report:
               precision    recall  f1-score   support

No Dysarthria       0.83      0.84      0.83      1025
   Dysarthria       0.71      0.70      0.70       579

     accuracy                           0.79      1604
    macro avg       0.77      0.77      0.77      1604
 weighted avg       0.79      0.79      0.79      1604



In [ ]:
#@title Baseline XG Boost

In [ ]:
xgb = XGBClassifier(
    objective='binary:logistic',  # Binary classification
    max_depth=3,  # Shallow trees to prevent overfitting
    n_estimators=50,  # Fewer trees
    learning_rate=0.05,  # Slower learning
    min_child_weight=3,  # Require more samples per leaf
    subsample=0.8,  # Row sampling
    colsample_bytree=0.8,  # Column sampling
    reg_alpha=1.0,  # L1 regularization
    reg_lambda=1.0,  # L2 regularization
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_scaled, y_train)
preds_xgb = xgb.predict(X_test_scaled)

print(f"Test Accuracy: {accuracy_score(y_test, preds_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds_xgb,
                          target_names=['No Dysarthria', 'Dysarthria']))

Test Accuracy: 0.9370

Classification Report:
               precision    recall  f1-score   support

No Dysarthria       0.93      0.98      0.95      1025
   Dysarthria       0.96      0.86      0.91       579

     accuracy                           0.94      1604
    macro avg       0.94      0.92      0.93      1604
 weighted avg       0.94      0.94      0.94      1604



In [ ]:
#@title Basic CNN

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow import keras

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Reshaped: {X_train.shape}")

Reshaped: (6415, 128, 1)


In [ ]:
#@title 1-Layer CNN
model = keras.Sequential([
    keras.layers.Conv1D(16, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=1)

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['No Dysarthria', 'Dysarthria']))

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


401/401 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.8717 - loss: 0.3161
Epoch 2/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9739 - loss: 0.0858
Epoch 3/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9823 - loss: 0.0649
Epoch 4/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9883 - loss: 0.0448
Epoch 5/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9881 - loss: 0.0410
Epoch 6/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9877 - loss: 0.0399
Epoch 7/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9912 - loss: 0.0320
Epoch 8/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9902 - loss: 0.0329
Epoch 9/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9923 - loss: 0.0262
Epoch 10/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9948 - loss: 0.0224
Epoch 11/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9941 - loss: 0.0231
Epoch 12/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accura

In [ ]:
#@title 3-Layer CNN
model = keras.Sequential([
    keras.layers.Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=30, batch_size=16, verbose=1)

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['No Dysarthria', 'Dysarthria']))

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


401/401 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8313 - loss: 0.3685
Epoch 2/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9641 - loss: 0.1180
Epoch 3/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9721 - loss: 0.0893
Epoch 4/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9813 - loss: 0.0634
Epoch 5/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9842 - loss: 0.0555
Epoch 6/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9799 - loss: 0.0575
Epoch 7/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9849 - loss: 0.0494
Epoch 8/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9865 - loss: 0.0449
Epoch 9/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9871 - loss: 0.0413
Epoch 10/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9854 - loss: 0.0421
Epoch 11/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9885 - loss: 0.0357
Epoch 12/30
401/401 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accurac

#  Unsupervised Clustering of Baseline Whisper Transcription using TF-IDF



In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
baseline_whisper_df = pd.read_csv('drive/MyDrive/baseline_whisper_preds.csv')

In [ ]:
# Extract features
X_text = baseline_whisper_df['hypothesis'].fillna("")
print(f"Total samples: {len(X)}")

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X_text)
print(f"TF-IDF features shape: {X_tfidf.shape}")

Total samples: 264
TF-IDF features shape: (264, 1000)


In [ ]:
# KMeans Clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_tfidf)
baseline_whisper_df['cluster'] = clusters

print(f"\nCluster distribution: {pd.Series(clusters).value_counts().sort_index().to_dict()}")
for i in range(3):
    print(f"\nCluster {i} sample:")
    print(f"  \"{baseline_whisper_df[baseline_whisper_df['cluster']==i]['hypothesis'].iloc[0][:80]}...\"")


Cluster distribution: {0: 8, 1: 124, 2: 132}

Cluster 0 sample:
  " Okay...."

Cluster 1 sample:
  " It was still a challenge...."

Cluster 2 sample:
  " You can write next...."


# Unsupervised Clustering of Finetuned Whisper Transcription using TF-IDF

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
finetuned_whisper_df = pd.read_csv('drive/MyDrive/finetuned_whisper_preds.csv')

In [ ]:
# Extract features
X_text = finetuned_whisper_df['hypothesis'].fillna("")
print(f"Total samples: {len(X)}")

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X_text)
print(f"TF-IDF features shape: {X_tfidf.shape}")

Total samples: 264
TF-IDF features shape: (292, 1000)


In [ ]:
# KMeans Clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_tfidf)
finetuned_whisper_df['cluster'] = clusters

print(f"\nCluster distribution: {pd.Series(clusters).value_counts().sort_index().to_dict()}")
for i in range(3):
    print(f"\nCluster {i} sample:")
    print(f"  \"{finetuned_whisper_df[finetuned_whisper_df['cluster']==i]['hypothesis'].iloc[0][:80]}...\"")


Cluster distribution: {0: 22, 1: 21, 2: 249}

Cluster 0 sample:
  " So ‡ okay ...."

Cluster 1 sample:
  " mhm ...."

Cluster 2 sample:
  " alright ...."


# End to end classification pipeline

In [ ]:
PARTICIPANT_NUMS = ["1554", "1713", "1731", "1738", "1833", "1944"]
MAX_AUDIO_ARRAY_LEN = 30 * 16000

In [ ]:
#@title Make dataset

import random

participant_num = PARTICIPANT_NUMS[random.randint(0, len(PARTICIPANT_NUMS))]

audio_array = librosa.load(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/video.mp4')
audio_array = librosa.resample(audio_array[0], orig_sr=audio_array[1], target_sr=16000)

dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/dataset.json'
data = []
i = 0
while i * MAX_AUDIO_ARRAY_LEN < len(audio_array):
  data.append({"i": i, "audio": audio_array[i*MAX_AUDIO_ARRAY_LEN:min(len(audio_array), (i+1)*MAX_AUDIO_ARRAY_LEN)].tolist()})
  i += 1

with open(dataset_output_path, 'w') as f:
  json.dump({"data": data}, f)

/tmp/ipython-input-2439631253.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array = librosa.load(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/video.mp4')
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model_checkpoint = "drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/finetuning/checkpoint-1200"

In [ ]:
#@title Set up model, tokenizer, and feature extractor
from transformers import WhisperForConditionalGeneration

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = WhisperForConditionalGeneration.from_pretrained(model_checkpoint)
model.to(device)

tokenizer = WhisperTokenizer.from_pretrained(model_checkpoint)
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/finetuning/checkpoint-1200'. Use `repo_type` argument if needed.

In [ ]:
## Code is based on model card for Whispr on HuggingFace (https://huggingface.co/openai/whisper-large-v3)
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])


Batching examples:   0%|          | 0/117 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:655: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 

In [ ]:
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('229-open-ai-api-key')

client = OpenAI()

def speaker_diarization(transcript):
  response = client.responses.create(
      model="gpt-5.1",
      instructions="You are given a transcript between and interviewer and a participant. Your task is toannotate the transcript to add a speaker tag before each speaker begins talking. The interviewer's speaker tag is *INV and the participant's speaker tag is *PAR. Return only the annotated transcript with no other changes",
      input=transcript
  )

  return response.output_text

In [ ]:
print(sorted_results)

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

## Transcript feature extraction

- Utterance metrics: Total_Utts, MLU_Utts (mean length of utterance from morphemes), MLU_Words (mean length of utterance in words), MLU_Morphemes
- Lexical metrics: FREQ_types, FREQ_tokens, FREQ_TTR, Words_Min
- Grammatical features: %_Nouns, %_Verbs, %_Aux, %_Mod, %_PAST, %_PRESP, %_prep, %_adj, %_adv, etc.
- Error metrics: %_Word_Errors, Utt_Errors, retracing, repetition
- Other features: density, noun_verb, open_closed, #open-class, #closed-class

From https://talkbank.org/0info/manuals/CLAN.pdf


In [ ]:
#@title Util functions for getting utterance metrics
def get_all_utterances(transcript, speaker):
  all_utterances = []
  for line in transcript.split("\n"):
    if not speaker in line:
      continue

    all_utterances.extend(line[4:].split(" ."))
  return all_utterances

def get_utterance_features(transcript, speaker):
  all_utterances = get_all_utterances(transcript, speaker)

  total_utterances = len(all_utterances)

  utterance_words = []
  for utt in all_utterances:
    utterance_words.extend(utt.split(" "))

  return {
      "total_utts": total_utterances,
      "mlu_utts": None,
      "mlu_words": None
  }

In [ ]:
#@title Util functions for getting lexical features
def get_all_words(transcript, speaker):
  all_words = []
  for line in transcript.split("\n"):
    if not speaker in line:
      continue

    utterances = line[4:].split(" .")
    for utt in utterances:
      all_words.extend(utt.split(" "))
  return all_words

def get_unique_words(all_words):
  return list(set(all_words))

def get_lexical_features(transcript, dataset, speaker):
  all_words = get_all_words(transcript, speaker)
  unique_words = get_unique_words(all_words)

  # length of the recording in minutes, audio is sampled at 16kHz
  audio_len = np.sum(np.array([len(audio) for audio in dataset["audio"]]))
  length = int(audio_len) / 16000 / 60

  return {
      "FREQ_types": len(unique_words),
      "FREQ_tokens": len(all_words),
      "FREQ_TTR": len(unique_words) / len(all_words),
      "Words_Min": len(all_words) / length
  }


In [ ]:
#@title Util functions for extracting grammatical features
from openai import OpenAI
from google.colab import userdata
from pydantic import BaseModel

os.environ["OPENAI_API_KEY"] = userdata.get('229-open-ai-api-key')

client = OpenAI()

class GrammaticalTypes(BaseModel):
  nouns: int
  verbs: int
  adjectives: int
  adverbs: int
  present_participles: int
  past_participles: int
  past_tense: int
  auxiliaries: int
  modals: int
  pronouns: int
  third_person_singular: int
  first_person_singular: int
  prepositions: int
  conjunctions: int
  determiners: int
  plurals: int

  def to_obj(self):
    return {
        "nouns": self.nouns,
        "verbs": self.verbs,
        "adj": self.adjectives,
        "adv": self.adverbs,
        "PresP": self.present_participles,
        "PastP": self.past_participles,
        "Past": self.past_tense,
        "aux": self.auxiliaries,
        "mod": self.modals,
        "pro": self.pronouns,
        "3S": self.third_person_singular,
        "prep": self.prepositions,
        "conj": self.conjunctions,
        "det": self.determiners,
        "plurals": self.plurals
    }

def get_percent_classes(grammatical_types_obj, all_words):
  result = {}
  for key, val in grammatical_types_obj.items():
    result[f'%_{key}'] = val / len(all_words)
  return result

OPEN_CLASSES = ["nouns", "adv", "adj", "verbs"]
OPEN_EXCLUDED = ["aux", "mod"]
def get_grammatical_features(transcript, speaker):
  all_words = get_all_words(transcript, speaker)
  grammatical_types = client.responses.parse(
      model="gpt-5.1",
      instructions="You are given a list of words, extract number of words of each grammatical type.",
      input=transcript,
      text_format=GrammaticalTypes
  ).output_parsed

  types_obj = grammatical_types.to_obj()

  result = get_percent_classes(types_obj, all_words)

  open_class_words = int(np.sum(np.array([types_obj[class_type] for class_type in OPEN_CLASSES])) - np.sum(np.array([types_obj[class_type] for class_type in OPEN_EXCLUDED])))
  closed_class_words = len(all_words) - open_class_words

  density_num_classes = ["verbs", "adjectives", "adverbs", "prepositions", "conjunctions"]
  density = int(np.sum(np.array([types_obj[class_type] for class_type in density_num_classes]))) / len(all_words)

  result["noun_verb"] = types_obj["nouns"] / types_obj["verbs"]
  result["open_closed"] = open_class_words / closed_class_words
  result["#open-class"] = open_class_words
  result["#closed-class"] = closed_class_words
  result["density"] = density

  return result

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)


NameError: name 'diarized_transcript' is not defined

# Generate

In [ ]:
#@title Make dataset

import random

for participant_num in PARTICIPANT_NUMS:
  print("Begin Participant", participant_num)
  audio_array = librosa.load(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/video.mp4')
  audio_array = librosa.resample(audio_array[0], orig_sr=audio_array[1], target_sr=16000)

  dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/dataset.json'

  data = []
  i = 0
  while i * MAX_AUDIO_ARRAY_LEN < len(audio_array):
    data.append({"i": i, "audio": audio_array[i*MAX_AUDIO_ARRAY_LEN:min(len(audio_array), (i+1)*MAX_AUDIO_ARRAY_LEN)].tolist()})
    i += 1

  with open(dataset_output_path, 'w') as f:
    json.dump({"data": data}, f)

  print("End Participant", participant_num)

Begin Participant 1944


/tmp/ipython-input-3728532962.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array = librosa.load(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{participant_num}/video.mp4')
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


End Participant 1944


In [ ]:
#@title Set up model, tokenizer, and feature extractor
from transformers import WhisperForConditionalGeneration

model_checkpoint = "drive/MyDrive/cs229_final_proj/finetuning/checkpoint-1200"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = WhisperForConditionalGeneration.from_pretrained(model_checkpoint)
model.to(device)

tokenizer = WhisperTokenizer.from_pretrained(model_checkpoint)
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('229-open-ai-api-key')

client = OpenAI()

def speaker_diarization(transcript):
  response = client.responses.create(
      model="gpt-5.1",
      instructions="You are given a transcript between and interviewer and a participant. Your task is toannotate the transcript to add a speaker tag before each speaker begins talking. The interviewer's speaker tag is *INV and the participant's speaker tag is *PAR. Return only the annotated transcript with no other changes",
      input=transcript
  )

  return response.output_text

# 1713

In [ ]:
dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1713"}/dataset.json'
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])
print(sorted_results)

`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50359, 50360, 50361, 50362, 50363], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTok

[{'predictions': [' Okay .', ' Oh okay. Could you tell me about it ?', " um well ‡ um I woke up and um um sit up and um it didn't feel strange or anything and I got up to go to the bathroom &-laughs and I couldn't get my arm &-ges to do something and I  I  d it didn't really dawn on me „ you know . So I came out and tried to make a  a coffee .", " I just couldn't make it &//laughs and I uh turned around and I fell and I didn't understand why I fell and um I went back in the room . My um granddaughter was spending the night and um I decided that um um I needed some help you-know and um"], 'id': [0, 1, 2, 3]}, {'predictions': [" I tried to call my son but I didn't  I didn't work you-know and my daughter came up and I d called her and all I could say was yeah  yeah  ya you-know that's all I could say um um and um then after I s fell again &-laughs and my daughter um I mean my granddaughter um", " uh took my phone and ran out here and called her dad you know . So that was you know the  got

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1713"}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1713"}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)

{'%_nouns': 0.10063559322033898, '%_verbs': 0.08686440677966102, '%_adj': 0.00847457627118644, '%_adv': 0.02065677966101695, '%_PresP': 0.011122881355932203, '%_PastP': 0.003707627118644068, '%_Past': 0.04396186440677966, '%_aux': 0.03919491525423729, '%_mod': 0.005296610169491525, '%_pro': 0.06991525423728813, '%_3S': 0.019067796610169493, '%_prep': 0.04449152542372881, '%_conj': 0.06673728813559322, '%_det': 0.06726694915254237, '%_plurals': 0.019597457627118644, 'noun_verb': 1.1585365853658536, 'open_closed': 0.20793346129238643, '#open-class': 325, '#closed-class': 1563, 'FREQ_types': 447, 'FREQ_tokens': 1888, 'FREQ_TTR': 0.2367584745762712, 'Words_Min': 52.08133087386053, 'total_utts': 171, 'mlu_utts': None, 'mlu_words': None}


# 1731

In [ ]:
dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1731"}/dataset.json'
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

In [ ]:
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])
print(sorted_results)

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:655: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


[{'predictions': [" okay ‡ so ‡ first we're just gonna do some talking .", ' um Jennifer  Jennifer . Oh mhm . Yeah , Jennifer . Speaking . Mhm . I know her . Mhm . Ruh really ? Mhm . Miss you a lot . Aww . Yeah . Um so ‡ okay . Do you remember when you had your stroke ? Yes indeed . Mhm .', ' .', " mhm . Or just for  but uh flying helicopters . mhm . Two weeks . mhm . Sleeping . mhm . mhm . That's really hard . Yeah ‡ right . But what can you do ? mhm . So ‡ what about after that ? What happened next ? Tutingen Germany uh ."], 'id': [0, 1, 2, 3]}, {'predictions': [' This one um...', ' god and uh Saudi and uh Felix and Tamra and  &//ges and sleeping and flying Germany and um uh what can you do but uh &//ges and then &//ges the  the  the  the  the and cut off  cut off the cut off . ', ' & .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .).) .)

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1731"}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1731"}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)

{'%_nouns': 0.1050012440905698, '%_verbs': 0.06170689226175666, '%_adj': 0.009703906444389152, '%_adv': 0.012689723811893506, '%_PresP': 0.02139835780044787, '%_PastP': 0.001492908683752177, '%_Past': 0.01443145060960438, '%_aux': 0.02413535705399353, '%_mod': 0.002736999253545658, '%_pro': 0.02289126648420005, '%_3S': 0.005971634735008708, '%_prep': 0.02886290121920876, '%_conj': 0.03483453595421747, '%_det': 0.039313262005474, '%_plurals': 0.01567554117939786, 'noun_verb': 1.7016129032258065, 'open_closed': 0.19364419364419364, '#open-class': 652, '#closed-class': 3367, 'FREQ_types': 630, 'FREQ_tokens': 4019, 'FREQ_TTR': 0.1567554117939786, 'Words_Min': 53.98496171587733, 'total_utts': 404, 'mlu_utts': None, 'mlu_words': None}


# 1738

In [ ]:
dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1738"}/dataset.json'
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

In [ ]:
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])
print(sorted_results)

[{'predictions': [' okay .', ' I can for the last seven and uh at times uh this tenth year I can wake up and feel fine and pronounce things very carefully .', ' uh yeah sure .', " uh behind my  uh a front of my uh doors and um I uh suddenly went uh I would say at this point I'll say blank but uh going back as far beyond this I couldn't get a hold myself and uh w I was uh"], 'id': [0, 1, 2, 3]}, {'predictions': [" It's  and the uh.).) house is very cold . It's very messy . And I sat for from some  between two and three days „ so ‡ wow ‡ okay .", ' Yeah  yeah . And so how has your uh recovery been since then and what kinds of things have you done to try to get better since your stroke ?', " I couldn't walk for about I think about four months  . Three or four months . But uh I'm uh have uh been doing the things I was asked to and uh and I read and I write and uh I draw and I walk w deal .", " Oh wow ‡ that's awesome . I love yoga too . I love yoga too ."], 'id': [4, 5, 6, 7]}, {'predictio

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1738"}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1738"}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)

{'%_nouns': 0.14285714285714285, '%_verbs': 0.11787072243346007, '%_adj': 0.021727322107550243, '%_adv': 0.011406844106463879, '%_PresP': 0.021184139054861488, '%_PastP': 0.010863661053775122, '%_Past': 0.035850081477457905, '%_aux': 0.042911461162411735, '%_mod': 0.005431830526887561, '%_pro': 0.06518196632265073, '%_3S': 0.018468223791417708, '%_prep': 0.06355241716458447, '%_conj': 0.049972840847365564, '%_det': 0.06735469853340575, '%_plurals': 0.019011406844106463, 'noun_verb': 1.2119815668202765, 'open_closed': 0.3254139668826494, '#open-class': 452, '#closed-class': 1389, 'FREQ_types': 508, 'FREQ_tokens': 1841, 'FREQ_TTR': 0.2759369907658881, 'Words_Min': 47.045187134359395, 'total_utts': 152, 'mlu_utts': None, 'mlu_words': None}


# 1833

In [ ]:
dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1833"}/dataset.json'
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

In [ ]:
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])
print(sorted_results)

[{'predictions': [" So first .).) so ‡ first I'm just gonna be asking you to do some talking . So how do you think your speech is these days ? Not very good . That's  that's my take on it . How do you mean ? I can't think  I can't  .", ' hm hm', " that's okay .", ' Simple words .'], 'id': [0, 1, 2, 3]}, {'predictions': [" I won't be al able to spell it . But  uh but to think of it I can't . Mhm . So is it easier for you to think of the word to say it than to write it or it's about the same ? Uh even  um I sti still struggle to text . Mhm . Because I can't think of the w uh write uh .", " Spelling and  &//laughs and uh uh word phrase . Do you use speech to text ever ? Does that help or is it still &//ges ? The  uh uh she w gets aggravated &//laughs so I don't do that .", ' You had your stroke . Oh yeah . Um could you tell me about it ? Well  well ‡ the  the initial one um I remember um we always kiss before we uh go to bed and uh uh well sh she s said uh', " did bu a pucker it was just 

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1833"}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1833"}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)

{'%_nouns': 0.15551743853630645, '%_verbs': 0.137221269296741, '%_adj': 0.04059462550028588, '%_adv': 0.022870211549456832, '%_PresP': 0.024013722126929673, '%_PastP': 0.010291595197255575, '%_Past': 0.07775871926815323, '%_aux': 0.05546026300743282, '%_mod': 0.008576329331046312, '%_pro': 0.08747855917667238, '%_3S': 0.036020583190394515, '%_prep': 0.07204116638078903, '%_conj': 0.09433962264150944, '%_det': 0.09491137793024586, '%_plurals': 0.030874785591766724, 'noun_verb': 1.1333333333333333, 'open_closed': 0.41276252019386106, '#open-class': 511, '#closed-class': 1238, 'FREQ_types': 440, 'FREQ_tokens': 1749, 'FREQ_TTR': 0.25157232704402516, 'Words_Min': 37.71437212187683, 'total_utts': 155, 'mlu_utts': None, 'mlu_words': None}


# 1944

In [ ]:
dataset_output_path = f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1944"}/dataset.json'
dataset = load_dataset("json", data_files=dataset_output_path, field="data", split="train")

In [ ]:
results = []
for batch in dataset.batch(batch_size=4):
  audio_arrays = batch["audio"]
  features = feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").input_features.to(device)
  with torch.no_grad():
    prediction_ids = model.generate(inputs=features)
  predictions = tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)
  results.append({"predictions": predictions, "id": batch["i"]})

sorted_results = sorted(results, key=lambda x: x["id"])
print(sorted_results)

Batching examples:   0%|          | 0/114 [00:00<?, ? examples/s]

[{'predictions': [' Så først vil jeg spørre deg om du kan ta noen taler .', " and couldn't say anything . And then I went up and up &-and tu took therapy  speech therapy and went u and I got  got it back . And uh now I  but the second stroke I  I have  I have some times when I wanna say something and I can't say it .", " uh way away ‡ oh yeah ‡ I  I said it &//laughs youknow . So I hadta  hadta work on that „ hm yeah ‡ understandable . So ‡ I know you're tellin us a little bit earlier , but I didn't get to hear most of it . Um do you remember when you had your stroke ? I know you've had two you can talk about either .", " st I will  my first stroke I didn't  ha I didn't know what it was but I knew I was feelin.) you-know s uneasy and  and stuff and  and then I  r it  it came on me and uh on my second stroke I  I .) I  I figured it was uh another one ."], 'id': [0, 1, 2, 3]}, {'predictions': [' Og så um I knew.).) some other things I had.).) to go do . It was the same . So I .).) I  I  

In [ ]:
full_transcript = " ".join([" ".join(r["predictions"]) for r in sorted_results])
diarized_transcript = speaker_diarization(full_transcript)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1944"}/full_transcript.txt', 'w') as f:
  f.write(diarized_transcript)

In [ ]:
import csv

grammatical_features = get_grammatical_features(diarized_transcript, "*PAR")
lexical_features = get_lexical_features(diarized_transcript, dataset, "*PAR")
utterance_features = get_utterance_features(diarized_transcript, "*PAR")

all_features = grammatical_features | lexical_features | utterance_features
print(all_features)

with open(f'drive/MyDrive/{APROCSA_DATASET_LOC}/{"1944"}/output_features.csv', 'w') as f:
  headers = list(all_features.keys())
  writer = csv.DictWriter(f, fieldnames=headers)

  writer.writeheader()
  writer.writerow(all_features)

{'%_nouns': 0.12839637719765584, '%_verbs': 0.09776238678742674, '%_adj': 0.018114011720831113, '%_adv': 0.014384656366542355, '%_PresP': 0.015982951518380393, '%_PastP': 0.008524240809802876, '%_Past': 0.038359083644112946, '%_aux': 0.04155567394778902, '%_mod': 0.005061267980820458, '%_pro': 0.05887053809270112, '%_3S': 0.020511454448588172, '%_prep': 0.03782631859350027, '%_conj': 0.05993606819392648, '%_det': 0.05966968566862014, '%_plurals': 0.018913159296750134, 'noun_verb': 1.3133514986376023, 'open_closed': 0.2691007437457742, '#open-class': 796, '#closed-class': 2958, 'FREQ_types': 537, 'FREQ_tokens': 3754, 'FREQ_TTR': 0.14304741608950453, 'Words_Min': 66.03904554169742, 'total_utts': 252, 'mlu_utts': None, 'mlu_words': None}


# End to end classification

## Utils

In [ ]:
overall_communication_impairment = {
    1738: 2,
    1944: 2,
    1713: 1,
    1554: 2,
    1833: 2,
    1731: 3
}

In [ ]:
import pandas as pd

In [ ]:
#@title Train data loading utils

def load_data():
  csv_path = f'drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/clan_features_with_labels.csv'
  return pd.read_csv(csv_path)

def train_and_test_participant_splits():
  random_participant_inds = np.random.choice(range(len(PARTICIPANT_NUMS)), size=2)
  test_participants = [int(PARTICIPANT_NUMS[i]) for i in random_participant_inds]
  train_participants = [int(PARTICIPANT_NUMS[i]) for i in range(len(PARTICIPANT_NUMS)) if i not in random_participant_inds]
  return (train_participants, test_participants)

EXTRACTED_FEATURES = ["%_Nouns", "%_Verbs", "%_adj", "%_adv",
        "%_PRESP", "%_PASTP", "%_PAST", "%_Aux", "%_Mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_Plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "Total_Utts", "Overall communication impairment"]
CLAN_COLS = ["%_Nouns", "%_Verbs", "%_adj", "%_adv",
        "%_PRESP", "%_PASTP", "%_PAST", "%_Aux", "%_Mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_Plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "Total_Utts"]
def filtered_train_dataset(df, train_participants):
  df = df[df["Participant"].isin(train_participants)]
  return df[EXTRACTED_FEATURES]

In [ ]:
#@title Test data loading utils
def load_test_data(test_participants):
  df = None
  for participant in test_participants:
    csv_path = f'drive/MyDrive/Stanford Classes/CS229/{APROCSA_DATASET_LOC}/{participant}/output_features.csv'
    dfi = pd.read_csv(csv_path)
    dfi["Participant"] = participant
    dfi["Overall communication impairment"] = overall_communication_impairment[participant]

    if df is None:
      df = dfi
    else:
      df = pd.concat([df, dfi])

  return df

COLUMN_ORDER = ["%_nouns", "%_verbs", "%_adj", "%_adv",
        "%_PresP", "%_PastP", "%_Past", "%_aux", "%_mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "total_utts"]
def filter_data_columns(df):
  return df[COLUMN_ORDER]


## Logistic Regression

In [ ]:
df = load_data()
train_participants, test_participants = train_and_test_participant_splits()
train_dataset = filtered_train_dataset(df, train_participants)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

clan_X = df[CLAN_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).values

scaler = StandardScaler()
X = scaler.fit_transform(clan_X)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = df["Overall communication impairment"].astype(int)

logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"  # important for small dataset
)

logreg.fit(X, y)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [ ]:
test_data = load_test_data(test_participants)

y_test = test_data["Overall communication impairment"].astype(int)
print(y_test)

test_data = filter_data_columns(test_data)
X_test = scaler.transform(test_data)

preds = logreg.predict(test_data)
print(preds)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

0    2
0    2
Name: Overall communication impairment, dtype: int64
[2 2]
Accuracy: 1.0
              precision    recall  f1-score   support

           2       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


## XGBoost

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = load_data()
train_participants, test_participants = train_and_test_participant_splits()
train_data = filtered_train_dataset(df, train_participants)

test_data = load_test_data(test_participants)

y_test = test_data["Overall communication impairment"].astype(int)

test_data = filter_data_columns(test_data)

In [ ]:
# Prepare labels (convert to 0-indexed)
y_train = df["Overall communication impairment"].astype(int) - 1

# Extract features
exclude = ['Overall communication impairment']

train_clan_cols = [c for c in train_data.columns if c not in exclude]
test_clan_cols = [c for c in test_data.columns if c not in exclude]

# CLAN features
clan_X_train = train_data[train_clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_test = test_data[test_clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

# Fit scaler on train only
scaler = StandardScaler()
clan_X_train_scaled = scaler.fit_transform(clan_X_train)
clan_X_test_scaled = scaler.transform(clan_X_test)

print(f"Dataset size: {len(df)} total")
print(f"Class distribution in test: {y.value_counts().sort_index().to_dict()}")

Dataset size: 54 total
Class distribution in test: {0: 9, 1: 36, 2: 9}


In [ ]:
# XGBoost model with heavy regularization
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    max_depth=2,
    n_estimators=30,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42
)
xgb.fit(clan_X_train_scaled, y_train)

XGBoostError: [03:55:19] /workspace/src/data/data.cc:557: Check failed: this->labels.Size() % this->num_row_ == 0 (18 vs. 0) : Incorrect size for labels: (54,1) v.s. 36
Stack trace:
  [bt] (0) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x2bdf8c) [0x7e48a80bdf8c]
  [bt] (1) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x5af3e0) [0x7e48a83af3e0]
  [bt] (2) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x5b10b0) [0x7e48a83b10b0]
  [bt] (3) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(XGDMatrixSetInfoFromInterface+0x10b) [0x7e48a7fc7a4b]
  [bt] (4) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x7e491da90e2e]
  [bt] (5) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x7e491da8d493]
  [bt] (6) /usr/lib/python3.12/lib-dynload/_ctypes.cpython-312-x86_64-linux-gnu.so(+0x98c1) [0x7e491ecc78c1]
  [bt] (7) /usr/lib/python3.12/lib-dynload/_ctypes.cpython-312-x86_64-linux-gnu.so(+0x8ffe) [0x7e491ecc6ffe]
  [bt] (8) /

In [ ]:
preds_xgb = xgb.predict(clan_X_test_scaled)
acc_xgb = accuracy_score(y_test, preds_xgb)
print(f"Test Accuracy: {acc_xgb:.3f}")
print(classification_report(y_test, preds_xgb))

ValueError: Feature shape mismatch, expected: 52, got 24

## Naive Bayes

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
import numpy as np

In [ ]:
df = load_data()
train_participants, test_participants = train_and_test_participant_splits()
train_data = filtered_train_dataset(df, train_participants)

test_data = load_test_data(test_participants)

y_test = test_data["Overall communication impairment"].astype(int)

test_data = filter_data_columns(test_data)

In [ ]:
# Prepare features
exclude = ['Overall communication impairment']

clan_cols = [c for c in df.columns if c not in exclude]
y = df["Overall communication impairment"].astype(int) - 1

print(f"Total dataset size: {len(df)}")
print(f"Class distribution:\n{y.value_counts().sort_index()}\n")

Total dataset size: 54
Class distribution:
Overall communication impairment
0     9
1    36
2     9
Name: count, dtype: int64



In [ ]:
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1,2))
text_X_train = vectorizer.fit_transform(df_train["transcript_text"])
text_X_test = vectorizer.transform(df_test["transcript_text"])

nb_text = MultinomialNB()
nb_text.fit(text_X_train, y_train)
preds_text = nb_text.predict(text_X_test)

print(f"Test Accuracy: {accuracy_score(y_test, preds_text):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_text))

Test Accuracy: 0.636

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.64      1.00      0.78         7
           2       0.00      0.00      0.00         2

    accuracy                           0.64        11
   macro avg       0.21      0.33      0.26        11
weighted avg       0.40      0.64      0.49        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
clan_X_full = train_data[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_full_scaled = StandardScaler().fit_transform(clan_X_full)

print("\n5-Fold CV Accuracy:")
cv_scores_clan = cross_val_score(
    GaussianNB(),
    clan_X_full_scaled,
    y,
    cv=5,
    scoring='accuracy'
)
print(f"Mean: {cv_scores_clan.mean():.3f} (+/- {cv_scores_clan.std():.3f})")
print(f"Individual folds: {cv_scores_clan}")

KeyError: "['File', 'Language', 'Corpus', 'Code', 'Age', 'Sex', 'Group', 'Role', 'Duration_(sec)', 'MLU_Utts', 'MLU_Words', 'MLU_Morphemes', 'Verbs_Utt', '%_Word_Errors', 'Utt_Errors', 'density', '%_13S', 'retracing', 'repetition', 'Participant', 'Word comprehension', 'Sentence comprehension', 'Word finding', 'Grammatical construction', 'Speech motor programming', 'Repetition', 'Reading', 'Overall'] not in index"

In [ ]:
X_train_combined = hstack([clan_X_train_scaled, text_X_train]).toarray()
X_test_combined = hstack([clan_X_test_scaled, text_X_test]).toarray()

nb_combined = GaussianNB()
nb_combined.fit(X_train_combined, y_train)
preds_combined = nb_combined.predict(X_test_combined)

print(f"Test Accuracy: {accuracy_score(y_test, preds_combined):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_combined))

Test Accuracy: 0.727

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.70      1.00      0.82         7
           2       1.00      0.50      0.67         2

    accuracy                           0.73        11
   macro avg       0.57      0.50      0.50        11
weighted avg       0.63      0.73      0.65        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# End to end classification 2

In [ ]:
EXTRACTED_FEATURES = ["%_Nouns", "%_Verbs", "%_adj", "%_adv",
        "%_PRESP", "%_PASTP", "%_PAST", "%_Aux", "%_Mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_Plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "Total_Utts", "Overall communication impairment"]

In [ ]:
#@title Load dataset
import pandas as pd

csv_path = f'drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/clan_features_with_labels.csv'
df = pd.read_csv(csv_path)

df = df[EXTRACTED_FEATURES]

df.head()

,%_Nouns,%_Verbs,%_adj,%_adv,%_PRESP,%_PASTP,%_PAST,%_Aux,%_Mod,%_pro,...,noun_verb,open_closed,#open-class,#closed-class,FREQ_types,FREQ_tokens,FREQ_TTR,Words_Min,Total_Utts,Overall communication impairment
0,25.000,15.385,0.000,3.846,12.500,12.500,50.000,3.846,0.000,5.769,...,1.625,0.821,23,28,29,52,0.558,NaN,6,2.0
1,17.600,19.200,1.600,7.200,4.167,4.167,58.333,0.800,0.800,14.400,...,0.917,0.838,57,68,62,125,0.496,NaN,15,2.0
2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,NaN,NaN,0,0,2,3,0.667,NaN,3,2.0
3,19.005,16.519,2.842,3.552,12.903,7.527,15.054,1.243,3.197,15.986,...,1.151,0.897,236,263,214,563,0.380,NaN,130,2.0
4,31.250,12.500,0.000,6.250,0.000,0.000,0.000,0.000,0.000,12.500,...,2.500,1.000,8,8,13,16,0.812,NaN,3,2.0


In [ ]:
#@title Baseline Logistic Regression

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]
clan_X = df[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

scaler = StandardScaler()
X = scaler.fit_transform(clan_X)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = df["Overall communication impairment"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"  # important for small dataset
)

logreg.fit(X_train, y_train)

preds = logreg.predict(X_test)

print("Validation accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Validation accuracy: 0.6363636363636364
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         2
           2       0.67      0.86      0.75         7
           3       1.00      0.50      0.67         2

    accuracy                           0.64        11
   macro avg       0.56      0.45      0.47        11
weighted avg       0.61      0.64      0.60        11



In [ ]:
random_participant_inds = np.random.choice(range(len(PARTICIPANT_NUMS)), size=2)
test_participants = [int(PARTICIPANT_NUMS[i]) for i in random_participant_inds]
train_participants = [int(PARTICIPANT_NUMS[i]) for i in range(len(PARTICIPANT_NUMS)) if i not in random_participant_inds]

df = None
for participant in test_participants:
  csv_path = f'drive/MyDrive/Stanford Classes/CS229/{APROCSA_DATASET_LOC}/{participant}/output_features.csv'
  dfi = pd.read_csv(csv_path)
  dfi["Participant"] = participant
  dfi["Overall communication impairment"] = overall_communication_impairment[participant]

  if df is None:
    df = dfi
  else:
    df = pd.concat([df, dfi])

y_test = df["Overall communication impairment"].astype(int)
print(y_test)

COLUMN_ORDER = ["%_nouns", "%_verbs", "%_adj", "%_adv",
        "%_PresP", "%_PastP", "%_Past", "%_aux", "%_mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "total_utts"]
def filter_data_columns(df):
  return df[COLUMN_ORDER]

df = filter_data_columns(df)

df

0    2
0    2
Name: Overall communication impairment, dtype: int64


,%_nouns,%_verbs,%_adj,%_adv,%_PresP,%_PastP,%_Past,%_aux,%_mod,%_pro,...,%_plurals,noun_verb,open_closed,#open-class,#closed-class,FREQ_types,FREQ_tokens,FREQ_TTR,Words_Min,total_utts
0,0.142857,0.117871,0.021727,0.011407,0.021184,0.010864,0.035850,0.042911,0.005432,0.065182,...,0.019011,1.211982,0.325414,452,1389,508,1841,0.275937,47.045187,152
0,0.122355,0.090804,0.008465,0.016160,0.017314,0.006926,0.030396,0.034629,0.004617,0.053482,...,0.016545,1.347458,0.247720,516,2083,508,2599,0.195460,44.765568,246


In [ ]:
x_test = df.apply(pd.to_numeric, errors='coerce').fillna(0).values

preds = logreg.predict(scaler.transform(x_test))
print("Test accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Test accuracy: 1.0
              precision    recall  f1-score   support

           2       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [ ]:
#@title Baseline XG Boost

import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Prepare labels (convert to 0-indexed)
y = df["Overall communication impairment"].astype(int) - 1

# Train/test split
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

df_train = df.loc[train_idx]
df_test = df.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# Extract features
exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]

# CLAN features
clan_X_train = df_train[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_test = df_test[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

# Fit scaler on train only
scaler = StandardScaler()
X_train = scaler.fit_transform(clan_X_train)
X_test = scaler.transform(clan_X_test)

print(f"Dataset size: {len(df)} total, {len(train_idx)} train, {len(test_idx)} test")
print(f"Class distribution in test: {y_test.value_counts().sort_index().to_dict()}")

Dataset size: 54 total, 43 train, 11 test
Class distribution in test: {0: 2, 1: 7, 2: 2}


In [ ]:
# XGBoost model with heavy regularization
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    max_depth=2,
    n_estimators=30,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42
)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, preds_xgb)
print(f"Validation Accuracy: {acc_xgb:.3f}")
print(classification_report(y_test, preds_xgb))

Validation Accuracy: 0.636
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.64      1.00      0.78         7
           2       0.00      0.00      0.00         2

    accuracy                           0.64        11
   macro avg       0.21      0.33      0.26        11
weighted avg       0.40      0.64      0.49        11



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
random_participant_inds = np.random.choice(range(len(PARTICIPANT_NUMS)), size=2)
test_participants = [int(PARTICIPANT_NUMS[i]) for i in random_participant_inds]
train_participants = [int(PARTICIPANT_NUMS[i]) for i in range(len(PARTICIPANT_NUMS)) if i not in random_participant_inds]

df = None
for participant in test_participants:
  csv_path = f'drive/MyDrive/Stanford Classes/CS229/{APROCSA_DATASET_LOC}/{participant}/output_features.csv'
  dfi = pd.read_csv(csv_path)
  dfi["Participant"] = participant
  dfi["Overall communication impairment"] = overall_communication_impairment[participant]

  if df is None:
    df = dfi
  else:
    df = pd.concat([df, dfi])

y_test = df["Overall communication impairment"].astype(int)
print(y_test)

COLUMN_ORDER = ["%_nouns", "%_verbs", "%_adj", "%_adv",
        "%_PresP", "%_PastP", "%_Past", "%_aux", "%_mod",
        "%_pro", "%_3S", "%_prep", "%_conj", "%_det",
        "%_plurals", "noun_verb", "open_closed", "#open-class", "#closed-class",
        "FREQ_types", "FREQ_tokens", "FREQ_TTR", "Words_Min", "total_utts"]
def filter_data_columns(df):
  return df[COLUMN_ORDER]

df = filter_data_columns(df)

df

0    2
0    1
Name: Overall communication impairment, dtype: int64


,%_nouns,%_verbs,%_adj,%_adv,%_PresP,%_PastP,%_Past,%_aux,%_mod,%_pro,...,%_plurals,noun_verb,open_closed,#open-class,#closed-class,FREQ_types,FREQ_tokens,FREQ_TTR,Words_Min,total_utts
0,0.128396,0.097762,0.018114,0.014385,0.015983,0.008524,0.038359,0.041556,0.005061,0.058871,...,0.018913,1.313351,0.269101,796,2958,537,3754,0.143047,66.039046,252
0,0.100636,0.086864,0.008475,0.020657,0.011123,0.003708,0.043962,0.039195,0.005297,0.069915,...,0.019597,1.158537,0.207933,325,1563,447,1888,0.236758,52.081331,171


In [ ]:
x_test = scaler.transform(df)
preds_xgb = xgb.predict(x_test)
acc_xgb = accuracy_score(y_test, preds_xgb)
print(f"Test Accuracy: {acc_xgb:.3f}")
print(classification_report(y_test, preds_xgb))

Test Accuracy: 0.500
              precision    recall  f1-score   support

           1       0.50      1.00      0.67         1
           2       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. U

In [ ]:
#@title Baseline Naive Bayes

In [ ]:
transcripts = {}
for participant in PARTICIPANT_NUMS:
  transcript = ""
  with open(f'drive/MyDrive/Stanford Classes/CS229/cs229_final_proj/aprocsa_dataset/{participant}/full_transcript.txt', 'r') as f:
    transcript = f.read()
  transcripts[int(participant)] = transcript


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
import numpy as np

In [ ]:
# Prepare features
exclude = [
    'File','Participant','Age','Sex','Group','Role',
    'Overall communication impairment','transcript_text'
]

clan_cols = [c for c in df.columns if c not in exclude]
y = df["Overall communication impairment"].astype(int) - 1

print(f"Total dataset size: {len(df)}")
print(f"Class distribution:\n{y.value_counts().sort_index()}\n")

# Split data
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=y, random_state=42
)

df_train = df.loc[train_idx]
df_test = df.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

Total dataset size: 54
Class distribution:
Overall communication impairment
0     9
1    36
2     9
Name: count, dtype: int64



In [ ]:
random_participant_inds = np.random.choice(range(len(PARTICIPANT_NUMS)), size=2)
test_participants = [int(PARTICIPANT_NUMS[i]) for i in random_participant_inds]
train_participants = [int(PARTICIPANT_NUMS[i]) for i in range(len(PARTICIPANT_NUMS)) if i not in random_participant_inds]

train_text_data = [transcripts[p] for p in train_participants]
test_text_data = [transcripts[p] for p in test_participants]


In [ ]:

vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1,2))
text_X_train = vectorizer.fit_transform(train_text_data)
text_X_test = vectorizer.transform(test_text_data)

y_train = [overall_communication_impairment[i] for i in train_participants]
y_test = [overall_communication_impairment[i] for i in test_participants]

nb_text = MultinomialNB()
nb_text.fit(text_X_train, y_train)
preds_text = nb_text.predict(text_X_test)

print(f"Test Accuracy: {accuracy_score(y_test, preds_text):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_text))

Test Accuracy: 0.000

Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00       2.0
           2       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
clan_X_full = df[clan_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
clan_X_full_scaled = StandardScaler().fit_transform(clan_X_full)

print("\n5-Fold CV Accuracy:")
cv_scores_clan = cross_val_score(
    GaussianNB(),
    clan_X_full_scaled,
    y,
    cv=5,
    scoring='accuracy'
)
print(f"Mean: {cv_scores_clan.mean():.3f} (+/- {cv_scores_clan.std():.3f})")
print(f"Individual folds: {cv_scores_clan}")


5-Fold CV Accuracy:
Mean: 0.515 (+/- 0.161)
Individual folds: [0.36363636 0.63636364 0.72727273 0.54545455 0.3       ]


In [ ]:
X_train_combined = hstack([clan_X_train_scaled, text_X_train]).toarray()
X_test_combined = hstack([clan_X_test_scaled, text_X_test]).toarray()

nb_combined = GaussianNB()
nb_combined.fit(X_train_combined, y_train)
preds_combined = nb_combined.predict(X_test_combined)

print(f"Test Accuracy: {accuracy_score(y_test, preds_combined):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, preds_combined))

NameError: name 'clan_X_train_scaled' is not defined